# KAN-83 E05 smoothing experiment report

## 1. Summary

The 28-day same-hour climatology blended equally with persistence had the lowest dev +24 h MAE (3.791), but its paired interval versus the smoother crosses zero. Beta 0.6 had a negative primary interval, fragile to block length.

## 2. Hypothesis and decision rule

The preregistrations hypothesize that smoothing or station same-hour history can improve +24 h MAE against the fixed smoother. Select the lowest valid dev MAE in each family and compare paired 72 h CIs; no holdout selection or promotion is made here.

## 3. Setup

Frozen snapshot `a00cfc9edd4c3fb5f7b946e8871be9e5a88fe2bb8ef56b91d895101da41c0208`; five weekly dev folds starting 2026-08-31 through 2026-09-28 UTC; expanding training, weekly refit at fold start minus 24 hours. Hourly observations assume availability at timestamp + 1 hour. Features use only timestamp <= origin - 1h and availability <= origin.

## 4. EDA and data checks

Five stations, 4,200 common +24 h pairs and 100% baseline-eligible coverage in each reported run. Air4Thai-era and KAN-74 cutoff exclusions are enforced by the shared runner. There are 15 observed spike pairs, so spike precision and recall are undefined under the 50-spike rule. Leakage, stale readings and smoother equivalence are covered by targeted tests.

## 5. Results



- Moving average: 24 h, MAE 3.926; CI vs smoother [-0.099, 0.154], run `3a4d8f968d104fe4818703cc29b28adb`.
- EWMA: 6 h half-life, MAE 3.874; CI [-0.061, 0.060], run `d2e50e44ba8343598ab50e5e75fcada4`.
- Beta smoother: beta 0.7, MAE 3.829; CI [-0.109, 0.008], run `3f8a628c51014da4ac62ffbbe58245ee`.
- Same-hour climatology: 28 days blended equally with persistence, MAE 3.791; CI [-0.339, 0.205], run `444eecf8cc5d4dc492e157ee0fa51104`.

Beta 0.6 has MAE 3.837 and a 72 h CI [-0.066, -0.007] below zero against the fixed smoother, but it is **fragile**: the 24 h sensitivity CI [-0.070, 0.002] crosses zero (168 h: [-0.051, -0.007]). Its dev result is not evidence of holdout performance or promotion. The lowest-MAE beta 0.7 and climatology blend have intervals crossing zero. Spike metrics are undefined: 15 observed spikes, below the minimum of 50. Pure climatology and the 50/50 blend use observations from the last N days at the target UTC hour, respecting the serving lag. EWMA weights all available history by elapsed hours. Beta 0.5 is the fixed smoother and beta 1.0 matches the 24 h moving average on this snapshot.



## 6. Statistical tests and diagnostics

Beta 0.6: 72 h CI [-0.066, -0.007], 24 h sensitivity [-0.070, 0.002], 168 h sensitivity [-0.051, -0.007]; fragile. Diebold-Mariano p=0.008 (167 h bandwidth), sensitivity p=0.084 (47 h), p=0.112 (23 h). Best climatology blend: 72 h CI [-0.339, 0.205], p=0.411. All dev intervals are unadjusted. There were 27 completed dev runs for 26 candidates; one moving-average candidate was repeated after optimization. Station promotion gate is a holdout decision and was not applied.

## 7. Figures

The chart compares the best valid candidate in each family using tracked +24 h MAE. The fixed smoother is included as a reference.

In [ ]:
import matplotlib.pyplot as plt
import mlflow

from pm25_forecast.experiments.tracking import tracking_uri

mlflow.set_tracking_uri(tracking_uri())
runs = {
    "MA 24h": "3a4d8f968d104fe4818703cc29b28adb",
    "EWMA 6h": "d2e50e44ba8343598ab50e5e75fcada4",
    "Beta 0.7": "3f8a628c51014da4ac62ffbbe58245ee",
    "Climatology blend 28d": "444eecf8cc5d4dc492e157ee0fa51104",
    "Smoother": "18ab9d50f90e4a9282d5dd04da06dfdb",
}
values = [mlflow.get_run(run).data.metrics["mae_24h"] for run in runs.values()]
plt.barh(list(runs), values)
plt.xlabel("+24 h MAE (µg/m³)")
plt.title("Best dev variant per family")
plt.show()

## 8. Conclusion, limitations, next

Do not promote based on these exploratory results. The lower observed MAE of the climatology blend is uncertain; beta 0.6 is sensitive to block size and falls short of the minimum-effect threshold. A later preregistered final-selection process must choose candidates without further holdout tuning. Results cover one season and the availability assumption is not observed arrival time.